# Query API Data

This notebook queries all data used in the project that comes from ENTSO-e Transparency Platform. 

The code is structured in the following sections: 

- **Imports and Settings**
- **Helper Functions for Downloading and Parsing**
- **Queries**
- **Initial Processing**
- **Eport**

In [ ]:
# =========================================================
# IMPORTS and Settings
# =========================================================

# ---------------------------------------------------------
# IMPORTS
# ---------------------------------------------------------

import requests
import pandas as pd
import pytz
import json
import time
import re

from datetime import datetime, timedelta


# ---------------------------------------------------------
# GLOBAL API CONFIGURATION
# ---------------------------------------------------------

BASE_URL = "https://transparency.entsoe.eu"

HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

MAX_RETRIES = 5
CHUNK_DAYS = 6


# ---------------------------------------------------------
# TIMEZONE CONFIG
# ---------------------------------------------------------

TZ_LOCAL = pytz.timezone("Europe/Vienna")
TZ_UTC = pytz.UTC

The helper functions include things like: 

- parsing ISO Durations
- wrapper for API query with retry-logic
- wrappers for iteration over timespans
- parsers for the differnt entpoints (including mappings)

In [ ]:
# =========================================================
# HELPER FUNCTIONS
# =========================================================

# ---------------------------------------------------------
# UTILITY: ISO DURATION PARSER
# ---------------------------------------------------------

def _parse_iso_duration(res):
    """
    Convert ISO8601 duration strings used by ENTSO-E
    (e.g. PT15M, PT60M, PT1H) into pandas Timedelta.
    """

    m = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?", res)

    hours = int(m.group(1) or 0)
    minutes = int(m.group(2) or 0)

    return pd.Timedelta(hours=hours, minutes=minutes)

# ---------------------------------------------------------
# UTILITY: RETRY POST REQUEST
# ---------------------------------------------------------

def _post_with_retry(endpoint, payload):
    """
    Send POST request to ENTSO-E API with retry logic.

    Retries up to MAX_RETRIES times if request fails.
    """

    url = BASE_URL + endpoint

    for attempt in range(MAX_RETRIES):

        try:

            r = requests.post(
                url,
                headers=HEADERS,
                data=json.dumps(payload),
                timeout=10
            )

            r.raise_for_status()

            return r.json()

        except requests.exceptions.RequestException as e:

            print(f"Attempt {attempt+1} failed: {e}")

            if attempt == MAX_RETRIES - 1:
                raise

            time.sleep(5)

# ---------------------------------------------------------
# UTILITY: QUERY TIME WINDOW
# ---------------------------------------------------------

def _generate_query_window(date_from, date_to):
    """
    Generate UTC query window for ENTSO-E queries.

    Adds one extra day before and after to avoid
    missing timestamps due to timezone conversion.
    """

    local_start = TZ_LOCAL.localize(datetime.strptime(date_from, "%Y-%m-%d"))
    local_end = TZ_LOCAL.localize(datetime.strptime(date_to, "%Y-%m-%d") + timedelta(days=1))

    utc_query_start = (local_start - timedelta(days=1)).astimezone(TZ_UTC)
    utc_query_end = (local_end + timedelta(days=1)).astimezone(TZ_UTC)

    return local_start, local_end, utc_query_start, utc_query_end

# ---------------------------------------------------------
# UTILITY: CHUNK GENERATOR
# ---------------------------------------------------------

def _chunk_range(start, end, chunck_days):
    """
    Generate 6-day query chunks for ENTSO-E API.

    API often fails for longer time ranges.
    """

    current = start

    while current < end:

        chunk_end = min(current + timedelta(days=chunck_days), end)

        yield current, chunk_end

        current = chunk_end

# ---------------------------------------------------------
# UTILITY: GENERIC PERIOD ITERATOR
# ---------------------------------------------------------

def _iterate_periods(data):
    """
    Iterate through ENTSO-E curveData → periodList → pointMap.

    Yields:
        instance metadata
        timestamp
        value container
    """

    for inst in data.get("instanceList", []):

        for period in inst.get("curveData", {}).get("periodList", []):

            start_utc = pd.to_datetime(period["timeInterval"]["from"], utc=True)

            res = _parse_iso_duration(period.get("resolution", "PT15M"))

            for idx_str, vals in period.get("pointMap", {}).items():

                try:
                    idx = int(idx_str)
                except:
                    continue

                ts = start_utc + idx * res

                yield inst, ts, vals

# ---------------------------------------------------------
# PARSER: META COLUMN DATASETS
# ---------------------------------------------------------

def parse_wind_solar_forecast(data):
    """
    Parse datasets that define columns in metaData.

    Used for:
        - wind forecast
        - solar forecast
    """

    meta = data.get("metaData", [])

    col_names = [m.get("code", f"col{i}") for i, m in enumerate(meta)]

    rows = []

    for inst, ts, vals in _iterate_periods(data):

        if not isinstance(vals, list):
            vals = [vals]

        vals = vals + [None]*(len(col_names)-len(vals))

        row = {"timestamp_utc": ts}

        for c,v in zip(col_names, vals):
            row[c] = v

        rows.append(row)

    return pd.DataFrame(rows)

# ---------------------------------------------------------
# PARSER: LOAD DATA
# ---------------------------------------------------------

def parse_load(data):
    """
    Parse load datasets.

    Extracts:
        load forecast
        load actual
    """

    rows = []

    for inst, ts, vals in _iterate_periods(data):

        forecast = None
        actual = None

        if isinstance(vals, list):

            if len(vals) > 0:
                forecast = vals[0]

            if len(vals) > 1:
                actual = vals[1]

        rows.append({
            "timestamp_utc": ts,
            "load_forecast_mw": forecast,
            "load_actual_mw": actual
        })

    return pd.DataFrame(rows)

# ---------------------------------------------------------
# PARSER: GENERATION BY TYPE
# ---------------------------------------------------------

def parse_generation_by_type(data):
    """
    Parse actual generation per production type.

    Converts PRODUCTION_TYPE dimension into dataframe columns.
    """

    mapping_generation = {
        "B01": "biomass_mw",                          
        "B02": "fossil_brown_coal_lignite_mw",        
        "B03": "fossil_coal_derived_gas_mw",
        "B04": "fossil_gas_mw",
        "B05": "fossil_hard_coal_mw",
        "B06": "fossil_oil_mw",
        "B07": "fossil_oil_shale_mw",
        "B08": "fossil_peat_mw",
        "B09": "geothermal_mw",
        "B10": "hydro_pumped_storage_mw",
        "B11": "hydro_run_of_river_and_poundage_mw",
        "B12": "hydro_water_reservoir_mw",
        "B13": "marine_mw",
        "B14": "nuclear_mw",
        "B15": "other_renewable_mw",
        "B16": "solar_mw",
        "B17": "waste_mw",
        "B18": "wind_offshore_mw",
        "B19": "wind_onshore_mw",
        "B20": "other_mw",
    }

    rows = []

    for inst, ts, vals in _iterate_periods(data):

        prod_type = inst.get(
            "businessDimensionMap", {}
        ).get("PRODUCTION_TYPE", "UNKNOWN")

        val = None

        if isinstance(vals, list) and len(vals) > 0:
            val = vals[0]
        else:
            val = vals

        if isinstance(val, dict):
            val = val.get("quantity")

        rows.append({
            "timestamp_utc": ts,
            "PRODUCTION_TYPE": mapping_generation.get(prod_type, prod_type),
            "value": val
        })

    df = pd.DataFrame(rows)

    return df.pivot(
        index="timestamp_utc",
        columns="PRODUCTION_TYPE",
        values="value"
    ).reset_index()

# ---------------------------------------------------------
# PARSER: SCHEDULED EXCHANGE
# ---------------------------------------------------------

def parse_scheduled_exchange(data):
    """
    Parse scheduled exchange flows between bidding zones.

    Extracts:
        IN_AREA
        OUT_AREA
        CONTRACT_TYPE
        exchange values
    """

    # Mapping for contract types
    CONTRACT_TYPE_MAP = {
        "A01": "day_ahead",
        "A05": "total"
    }

    # Mapping for areas
    AREA_MAP = {
        "BZN|10YAT-APG------L": "AT",
        "BZN|10YCH-SWISSGRIDZ": "CH",
        "BZN|10YCZ-CEPS-----N": "CZ",
        "BZN|10Y1001A1001A82H": "DE",
        "BZN|10YHU-MAVIR----U": "HU",
        "BZN|10Y1001A1001A73I": "IT",
        "BZN|10YSI-ELES-----O": "SI",
    }

    rows = []

    for inst, ts, vals in _iterate_periods(data):
        dims = inst.get("businessDimensionMap", {})

        out_area = dims.get("OUT_AREA")
        in_area = dims.get("IN_AREA")
        contract_type = dims.get("CONTRACT_TYPE")

        # Map codes
        out_area = AREA_MAP.get(out_area, out_area)
        in_area = AREA_MAP.get(in_area, in_area)
        contract_type = CONTRACT_TYPE_MAP.get(contract_type, contract_type)

        exchange = vals[0] if isinstance(vals, list) and vals else None

        if in_area and out_area and contract_type:
            col_name = f"{in_area}_to_{out_area}_{contract_type}"
            rows.append({
                "timestamp_utc": ts,
                col_name: exchange
            })

    # Combine all dicts into a single DataFrame efficiently
    df = pd.DataFrame(rows)
    
    # Group by timestamp, summing in case of duplicates
    df = df.groupby("timestamp_utc").agg("first").reset_index()

    return df

# ---------------------------------------------------------
# PARSER: DAY-AHEAD PRICES
# ---------------------------------------------------------

def parse_day_ahead_prices(data):
    """
    Parse day-ahead price data.

    Extracts:
        IN_AREA
        OUT_AREA
        CONTRACT_TYPE
        price values
    """

    rows = []

    for inst, ts, vals in _iterate_periods(data):

        price = vals[0] if isinstance(vals, list) and vals else None

        col_name = f"day_ahead_price"
        rows.append({
            "timestamp_utc": ts,
            col_name: price
        })

    df = pd.DataFrame(rows)
    
    # Group by timestamp, summing in case of duplicates
    df = df.groupby("timestamp_utc").agg("first").reset_index()

    return df

# ---------------------------------------------------------
# ENTSO-E ENDPOINT CONFIGURATION
# ---------------------------------------------------------

ENTSOE_ENDPOINTS = {

    "load": {
        "endpoint": "/load/total/dayAhead/load",
        "parser": parse_load
    },

    "solar_forecast": {
        "endpoint": "/generation/forecast/windAndSolar/solar/load",
        "parser": parse_wind_solar_forecast
    },

    "wind_onshore_forecast": {
        "endpoint": "/generation/forecast/windAndSolar/onshore/load",
        "parser": parse_wind_solar_forecast
    },

    "generation_by_type": {
        "endpoint": "/generation/actual/perType/generation/load",
        "parser": parse_generation_by_type
    },

    "scheduled_exchange": {
        "endpoint": "/transmission/scheduledExchange/load",
        "parser": parse_scheduled_exchange
    },
    "day_ahead_prices": {
        "endpoint": "/market/energyPrices/load",
        "parser": parse_day_ahead_prices
    }
}

# ----------------------------------------------------------
# MAIN QUERY FUNCTION
# ----------------------------------------------------------

def query_entsoe(dataset, date_from, date_to, area="10YAT-APG------L", chunk_days=CHUNK_DAYS):
    """
    Generic ENTSO-E query engine.

    Uses endpoint configuration and parser defined
    in ENTSOE_ENDPOINTS registry.
    """

    cfg = ENTSOE_ENDPOINTS[dataset]

    endpoint = cfg["endpoint"]
    parser = cfg["parser"]

    local_start, local_end, utc_start, utc_end = _generate_query_window(date_from, date_to)

    fmt = "%Y-%m-%dT%H:%M:%S.000Z"

    dfs = []

    for chunk_start, chunk_end in _chunk_range(utc_start, utc_end, chunk_days):

        print(f"Querying {dataset}: {chunk_start} → {chunk_end}")

        payload = {
            "dateTimeRange": {
                "from": chunk_start.strftime(fmt),
                "to": chunk_end.strftime(fmt)
            },
            "areaList": [f"BZN|{area}"],
            "timeZone": "UTC",
            "sorterList": [],
            "filterMap": {}
        }

        data = _post_with_retry(endpoint, payload)

        df = parser(data)

        dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    df = pd.concat(dfs, ignore_index=True)

    # --- timestamps ---
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
    df["timestamp_local"] = df["timestamp_utc"].dt.tz_convert(TZ_LOCAL)

    # --- set index ---
    df = df.set_index("timestamp_local").sort_index()

    # --- filter ---
    df = df[(df.index >= local_start) & (df.index < local_end)]

    # --- drop helper column ---
    df = df.drop(columns=["timestamp_utc"], errors="ignore")

    # --- enforce numeric types on all remaining columns ---
    df = df.apply(pd.to_numeric, errors="coerce")

    return df

The main data collection is performed here using the helper functions defined earlier. 

In [ ]:
# =========================================================
# DATA COLLECTION
# =========================================================

# Define query time range
from_date = "2019-01-01"
to_date = "2025-12-31"

# Query datasets for AT bidding zone
df_load = query_entsoe("load", from_date, to_date)
df_solar = query_entsoe("solar_forecast", from_date, to_date)
df_wind = query_entsoe("wind_onshore_forecast", from_date, to_date)
df_generation = query_entsoe("generation_by_type", from_date, to_date)
df_exchange = query_entsoe("scheduled_exchange", from_date, to_date)
df_prices = query_entsoe("day_ahead_prices", from_date, to_date, chunk_days=1)

# query prices neighboring bidding zones
df_prices_DE = query_entsoe("day_ahead_prices", from_date, to_date, area="10Y1001A1001A82H", chunk_days=1)
df_prices_CH = query_entsoe("day_ahead_prices", from_date, to_date, area="10YCH-SWISSGRIDZ", chunk_days=1)
df_prices_CZ = query_entsoe("day_ahead_prices", from_date, to_date, area="10YCZ-CEPS-----N", chunk_days=1)
df_prices_IT = query_entsoe("day_ahead_prices", from_date, to_date, area="10Y1001A1001A73I", chunk_days=1)
df_prices_HU = query_entsoe("day_ahead_prices", from_date, to_date, area="10YHU-MAVIR----U", chunk_days=1)
df_prices_SI = query_entsoe("day_ahead_prices", from_date, to_date, area="10YSI-ELES-----O", chunk_days=1)

Querying scheduled_exchange: 2018-12-30 23:00:00+00:00 → 2019-01-05 23:00:00+00:00
Querying scheduled_exchange: 2019-01-05 23:00:00+00:00 → 2019-01-11 23:00:00+00:00
Querying scheduled_exchange: 2019-01-11 23:00:00+00:00 → 2019-01-17 23:00:00+00:00
Querying scheduled_exchange: 2019-01-17 23:00:00+00:00 → 2019-01-23 23:00:00+00:00
Querying scheduled_exchange: 2019-01-23 23:00:00+00:00 → 2019-01-29 23:00:00+00:00
Querying scheduled_exchange: 2019-01-29 23:00:00+00:00 → 2019-02-04 23:00:00+00:00
Querying scheduled_exchange: 2019-02-04 23:00:00+00:00 → 2019-02-10 23:00:00+00:00
Querying scheduled_exchange: 2019-02-10 23:00:00+00:00 → 2019-02-16 23:00:00+00:00
Querying scheduled_exchange: 2019-02-16 23:00:00+00:00 → 2019-02-22 23:00:00+00:00
Querying scheduled_exchange: 2019-02-22 23:00:00+00:00 → 2019-02-28 23:00:00+00:00
Querying scheduled_exchange: 2019-02-28 23:00:00+00:00 → 2019-03-06 23:00:00+00:00
Querying scheduled_exchange: 2019-03-06 23:00:00+00:00 → 2019-03-12 23:00:00+00:00
Quer

Some renaming, resampling (qh and h) of the query results. Also joining of all the different data types. 

In [ ]:
# =========================================================
# INITIAL PROCESSING
# =========================================================

# rename price columns with bidding zone suffix
df_prices_DE = df_prices_DE.rename(columns={"day_ahead_price": "day_ahead_price_DE"})
df_prices_CH = df_prices_CH.rename(columns={"day_ahead_price": "day_ahead_price_CH"})
df_prices_CZ = df_prices_CZ.rename(columns={"day_ahead_price": "day_ahead_price_CZ"})
df_prices_IT = df_prices_IT.rename(columns={"day_ahead_price": "day_ahead_price_IT"})
df_prices_HU = df_prices_HU.rename(columns={"day_ahead_price": "day_ahead_price_HU"})
df_prices_SI = df_prices_SI.rename(columns={"day_ahead_price": "day_ahead_price_SI"})

# for wind and solar just keep cols "DAY_AHEAD" and "CURRENT" and rename to prefix with "solar_" and "wind_" 
df_solar = df_solar[["DAY_AHEAD", "CURRENT"]].rename(columns={"DAY_AHEAD": "solar_day_ahead", "CURRENT": "solar_current"})
df_wind = df_wind[["DAY_AHEAD", "CURRENT"]].rename(columns={"DAY_AHEAD": "wind_day_ahead", "CURRENT": "wind_current"})

# convert df_exchange from h ts to qh ts by taking the same value for each 15-min interval within the same hour
df_exchange_qh = df_exchange.resample("15min").ffill()

# convert every other df from qh to h averaging values within the same hour
df_load_h = df_load.resample("h").mean()
df_solar_h = df_solar.resample("h").mean()
df_wind_h = df_wind.resample("h").mean()
df_generation_h = df_generation.resample("h").mean()
df_prices_h = df_prices.resample("h").mean()
df_prices_DE_h = df_prices_DE.resample("h").mean()
df_prices_CH_qh = df_prices_CH.resample("15min").ffill()
df_prices_CZ_qh = df_prices_CZ.resample("15min").ffill()
df_prices_IT_qh = df_prices_IT.resample("15min").ffill()
df_prices_HU_qh = df_prices_HU.resample("15min").ffill()
df_prices_SI_qh = df_prices_SI.resample("15min").ffill()

# combine all h dataframes into a single dataframe
df_all_h = df_load_h.join(df_solar_h, how="outer") \
    .join(df_wind_h, how="outer") \
    .join(df_generation_h, how="outer") \
    .join(df_prices_h, how="outer") \
    .join(df_exchange, how="outer") \
    .join(df_prices_DE_h, how="outer") \
    .join(df_prices_CH, how="outer") \
    .join(df_prices_CZ, how="outer") \
    .join(df_prices_IT, how="outer") \
    .join(df_prices_HU, how="outer") \
    .join(df_prices_SI, how="outer")

# combine all qh dataframes into a single dataframe
df_all_qh = df_load.join(df_solar, how="outer") \
    .join(df_wind, how="outer") \
    .join(df_generation, how="outer") \
    .join(df_prices, how="outer") \
    .join(df_exchange_qh, how="outer") \
    .join(df_prices_DE, how="outer") \
    .join(df_prices_CH_qh, how="outer") \
    .join(df_prices_CZ_qh, how="outer") \
    .join(df_prices_IT_qh, how="outer") \
    .join(df_prices_HU_qh, how="outer") \
    .join(df_prices_SI_qh, how="outer")

Export into h and qh csvs

In [ ]:
# =========================================================
# EXPORT
# =========================================================

df_all_h.to_csv("../data/entsoe_data_h.csv")
df_all_qh.to_csv("../data/entsoe_data_qh.csv")